# MRIxFields2026 Task 3 — dual-T4 Kaggle baseline

This notebook runs the organizers' single shared multimodal StarGAN v2 checkpoint over all 180 validation translations. Two independent workers split the same deterministic case list 90/90 across Kaggle's two T4 GPUs. It validates every full volume, calls the current official Task 3 builder, validates all 180 cropped predictions, and creates `task3.zip`.

Before running: select **GPU T4 x2**, enable Internet for the official source/checkpoint checks, and attach private Kaggle datasets containing (1) this project's Task 3 helper files and (2) the released validation split. Edit only the two `CHANGE_ME` paths in the next cell. Expensive actions default to off, and nothing uploads externally.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import os
import shutil
import subprocess
import sys
import time

PROJECT_SOURCE = Path('/kaggle/input/CHANGE_ME-task3-code')
VALIDATION_DIR = Path('/kaggle/input/CHANGE_ME-mrixfields-data/validation_prospective')
ATTACHED_CHECKPOINT_VALUE = ''  # Optional: /kaggle/input/.../checkpoint_epoch50.pth

OFFICIAL_COMMIT = 'a22abccbbf61e2131d27331035ff5b9114565fbd'
HF_REPO_ID = 'mrixfields/MRIxFields2026-Baseline'
HF_REVISION = 'f1a7d6b741b64c81871d0856fa2d7295d564c141'
HF_FILE = 'task3_any_to_any_multimodal/stargan_v2/pro_pretrained/weights/checkpoint_epoch50.pth'
EXPECTED_CHECKPOINT_SIZE = 631346455
EXPECTED_CHECKPOINT_SHA256 = 'c2f8172c1fbdbf0426e22c7d3d2014632f175a38cf818f95e72b1917badabdeb'

WORK_ROOT = Path('/kaggle/working/mrixfields-task3')
CODE_DIR = WORK_ROOT / 'code'
OFFICIAL_REPO = WORK_ROOT / 'official' / 'MRIxFields2026'
WEIGHTS_DIR = WORK_ROOT / 'weights'
RESULTS_DIR = WORK_ROOT / 'results'
SMOKE_OUTPUT = RESULTS_DIR / 'smoke'
FULL_OUTPUT = RESULTS_DIR / 'full'
REPORT_DIR = RESULTS_DIR / 'reports'
SUBMISSION_DIR = RESULTS_DIR / 'submission_epoch50'
ARCHIVE = WORK_ROOT / 'task3.zip'
PIPELINE = CODE_DIR / 'task3_pipeline.py'
SUBMISSION_HELPER = CODE_DIR / 'task3_build_submission.py'
REQUIREMENTS = CODE_DIR / 'requirements-colab.txt'
CHECKPOINT = WEIGHTS_DIR / HF_FILE

if any('CHANGE_ME' in str(path) for path in (PROJECT_SOURCE, VALIDATION_DIR)):
    raise ValueError('Edit PROJECT_SOURCE and VALIDATION_DIR before continuing.')
if not PROJECT_SOURCE.is_dir():
    raise FileNotFoundError(PROJECT_SOURCE)
if not VALIDATION_DIR.is_dir():
    raise FileNotFoundError(VALIDATION_DIR)

for path in (CODE_DIR, WEIGHTS_DIR, RESULTS_DIR, REPORT_DIR):
    path.mkdir(parents=True, exist_ok=True)
for filename in ('task3_pipeline.py', 'task3_build_submission.py', 'requirements-colab.txt'):
    source = PROJECT_SOURCE / filename
    if not source.is_file():
        raise FileNotFoundError(f'Missing project helper: {source}')
    shutil.copy2(source, CODE_DIR / filename)

help_text = subprocess.check_output([sys.executable, str(PIPELINE), '--help'], text=True)
if 'finalize-shards' not in help_text:
    raise RuntimeError('The attached task3_pipeline.py is too old for dual-GPU sharding.')

env_text = '\n'.join((
    f'DATA_DIR={VALIDATION_DIR.parent}',
    f'PREPROCESSED_DIR={WORK_ROOT / "preprocessed_unused"}',
    f'OUTPUT_DIR={RESULTS_DIR}',
    f'INFERENCE_DIR={FULL_OUTPUT}',
    f'SUBMISSION_DIR={SUBMISSION_DIR}',
    f'PREDICTIONS_SEG_DIR={WORK_ROOT / "unused_task3_seg"}',
    'DEVICE=cuda:0',
)) + '\n'
(WORK_ROOT / '.env').write_text(env_text, encoding='utf-8')
os.environ['HF_HOME'] = str(WORK_ROOT / '.cache' / 'huggingface')
os.environ['TORCH_HOME'] = str(WORK_ROOT / '.cache' / 'torch')
print('Working root:', WORK_ROOT)
print('Validation input:', VALIDATION_DIR)

## Install the small inference dependency set and verify both GPUs

In [ ]:
subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '-q', '-r', str(REQUIREMENTS)
])

import torch

if not torch.cuda.is_available() or torch.cuda.device_count() < 2:
    raise RuntimeError(
        f'This notebook requires two GPUs; detected {torch.cuda.device_count()}.'
    )
gpu_names = [torch.cuda.get_device_name(index) for index in range(2)]
for index, name in enumerate(gpu_names):
    memory_gib = torch.cuda.get_device_properties(index).total_memory / 2**30
    print(f'GPU {index}: {name}, {memory_gib:.1f} GiB')
if not all('T4' in name for name in gpu_names):
    print('Warning: two GPUs are available, but they are not both reported as T4s.')

## Recheck and fetch the live official sources

This deliberately stops if GitHub or Hugging Face changes. Review the [competition page](https://mrixfields.chihucloud.com/2026/), [tasks page](https://mrixfields.chihucloud.com/2026/tasks.html), [validation submission README](https://github.com/MRIxFields/MRIxFields2026/blob/main/Submission/README.md), and separate [test-phase Docker guide](https://github.com/MRIxFields/MRIxFields2026/blob/main/Submission/testing-2026/doc/participant_guide.md) before the final build, then set `LIVE_RULES_REVIEWED = True`. This notebook builds the validation-phase 180-file `task3.zip`; it does not build the test-phase Docker image.

In [ ]:
LIVE_RULES_REVIEWED = False
LIVE_RULE_URLS = (
    'https://mrixfields.chihucloud.com/2026/',
    'https://mrixfields.chihucloud.com/2026/tasks.html',
    'https://github.com/MRIxFields/MRIxFields2026/blob/main/Submission/README.md',
    'https://github.com/MRIxFields/MRIxFields2026/blob/main/Submission/testing-2026/doc/participant_guide.md',
)

remote_line = subprocess.check_output([
    'git', 'ls-remote', 'https://github.com/MRIxFields/MRIxFields2026.git',
    'refs/heads/main',
], text=True).strip()
live_official_commit = remote_line.split()[0]
if live_official_commit != OFFICIAL_COMMIT:
    raise RuntimeError(
        f'Official main changed to {live_official_commit}; stop and review Task 3.'
    )

from huggingface_hub import HfApi
live_hf_revision = HfApi().repo_info(repo_id=HF_REPO_ID).sha
if live_hf_revision != HF_REVISION:
    raise RuntimeError(
        f'Hugging Face main changed to {live_hf_revision}; stop and review Task 3.'
    )

subprocess.check_call([
    sys.executable, str(PIPELINE), 'fetch-official',
    '--destination', str(OFFICIAL_REPO), '--revision', OFFICIAL_COMMIT,
])
revision = subprocess.check_output(
    ['git', '-C', str(OFFICIAL_REPO), 'rev-parse', 'HEAD'], text=True
).strip()
source_report = {
    'checked_at_utc': datetime.now(timezone.utc).isoformat(),
    'official_git_revision': revision,
    'huggingface_revision': live_hf_revision,
    'rules_reviewed': LIVE_RULES_REVIEWED,
    'rule_urls': list(LIVE_RULE_URLS),
}
(REPORT_DIR / 'live_source_check.json').write_text(
    json.dumps(source_report, indent=2) + '\n', encoding='utf-8'
)
print(source_report)
if not LIVE_RULES_REVIEWED:
    print('Final packaging remains locked until the live rules are reviewed.')

## Stage and verify the one official shared multimodal checkpoint

Leave `ATTACHED_CHECKPOINT_VALUE` empty to download the pinned file. To run without a checkpoint download, attach the official file as a private Kaggle dataset and set its exact path in the first cell.

In [ ]:
if ATTACHED_CHECKPOINT_VALUE:
    attached_checkpoint = Path(ATTACHED_CHECKPOINT_VALUE)
    if not attached_checkpoint.is_file():
        raise FileNotFoundError(attached_checkpoint)
    CHECKPOINT.parent.mkdir(parents=True, exist_ok=True)
    temporary = CHECKPOINT.with_name('.checkpoint_epoch50.pth.part')
    shutil.copy2(attached_checkpoint, temporary)
    os.replace(temporary, CHECKPOINT)
elif not CHECKPOINT.is_file():
    subprocess.check_call([
        sys.executable, str(PIPELINE), 'download-checkpoint',
        '--destination', str(WEIGHTS_DIR),
    ])

digest = hashlib.sha256()
with CHECKPOINT.open('rb') as stream:
    for chunk in iter(lambda: stream.read(8 * 1024 * 1024), b''):
        digest.update(chunk)
checkpoint_sha256 = digest.hexdigest()
if CHECKPOINT.stat().st_size != EXPECTED_CHECKPOINT_SIZE:
    raise RuntimeError(f'Unexpected checkpoint size: {CHECKPOINT.stat().st_size}')
if checkpoint_sha256 != EXPECTED_CHECKPOINT_SHA256:
    raise RuntimeError(f'Checkpoint SHA-256 mismatch: {checkpoint_sha256}')
checkpoint_report = {
    'repository': HF_REPO_ID, 'revision': HF_REVISION, 'filename': HF_FILE,
    'path': str(CHECKPOINT), 'size_bytes': CHECKPOINT.stat().st_size,
    'sha256': checkpoint_sha256,
}
(REPORT_DIR / 'checkpoint_verified.json').write_text(
    json.dumps(checkpoint_report, indent=2) + '\n', encoding='utf-8'
)
print(checkpoint_report)

## Audit the 45 released inputs and plan all 180 non-identity cases

In [ ]:
subprocess.check_call([
    sys.executable, str(PIPELINE), 'audit',
    '--input-root', str(VALIDATION_DIR),
    '--modalities', 'T1W', 'T2W', 'T2FLAIR', '--check-values',
    '--report', str(REPORT_DIR / 'input_audit.json'),
])
plan_text = subprocess.check_output([
    sys.executable, str(PIPELINE), 'plan',
    '--input-root', str(VALIDATION_DIR), '--output-dir', str(FULL_OUTPUT),
    '--modalities', 'T1W', 'T2W', 'T2FLAIR',
], text=True)
(REPORT_DIR / 'case_plan.txt').write_text(plan_text, encoding='utf-8')
print('\n'.join(plan_text.splitlines()[:4]))

## One full-volume smoke translation on GPU 0

Set the flag to `True` and run this before the full sweep. It verifies T1W `0.1T -> 1.5T`, subject `0001`, including joint target-domain index, shape, spatial metadata, dtype, finiteness, and range.

In [ ]:
RUN_PRETRAINED_SMOKE = False
if RUN_PRETRAINED_SMOKE:
    subprocess.check_call([
        sys.executable, str(PIPELINE), 'infer',
        '--input-root', str(VALIDATION_DIR), '--output-dir', str(SMOKE_OUTPUT),
        '--official-repo', str(OFFICIAL_REPO),
        '--official-revision', OFFICIAL_COMMIT, '--checkpoint', str(CHECKPOINT),
        '--modalities', 'T1W', '--device', 'cuda:0', '--batch-size', '1',
        '--seed', '20260712', '--smoke', '--resume',
    ])
    subprocess.check_call([
        sys.executable, str(PIPELINE), 'validate-outputs',
        '--input-root', str(VALIDATION_DIR), '--output-dir', str(SMOKE_OUTPUT),
        '--modalities', 'T1W', '--smoke',
        '--report', str(REPORT_DIR / 'smoke_validation.json'),
    ])
else:
    print('Smoke inference is OFF.')

## Complete dual-T4 sweep: 90 cases per GPU

Both processes load the same official checkpoint and use the same seed, config, and 15-domain mapping. `CUDA_VISIBLE_DEVICES` isolates one physical T4 per worker. Outputs are disjoint and resumable; rerunning validates and skips completed files. After both workers finish, `finalize-shards` verifies their provenance, validates all 180 full volumes, and creates the canonical manifest required by the submission checker.

In [ ]:
RUN_PRETRAINED_FULL_SWEEP = False
if RUN_PRETRAINED_FULL_SWEEP:
    processes = []
    for shard_index in range(2):
        command = [
            sys.executable, str(PIPELINE), 'infer',
            '--input-root', str(VALIDATION_DIR), '--output-dir', str(FULL_OUTPUT),
            '--official-repo', str(OFFICIAL_REPO),
            '--official-revision', OFFICIAL_COMMIT, '--checkpoint', str(CHECKPOINT),
            '--modalities', 'T1W', 'T2W', 'T2FLAIR',
            '--device', 'cuda:0', '--batch-size', '1', '--seed', '20260712',
            '--num-shards', '2', '--shard-index', str(shard_index), '--resume',
        ]
        worker_env = os.environ.copy()
        worker_env['CUDA_VISIBLE_DEVICES'] = str(shard_index)
        worker_env['PYTHONUNBUFFERED'] = '1'
        processes.append(subprocess.Popen(command, env=worker_env))
        print(f'Started shard {shard_index} on physical GPU {shard_index}')

    try:
        while any(process.poll() is None for process in processes):
            failed = [process.returncode for process in processes if process.poll() not in (None, 0)]
            if failed:
                for process in processes:
                    if process.poll() is None:
                        process.terminate()
                raise RuntimeError(f'A dual-GPU worker failed: {failed}')
            time.sleep(2)
    except KeyboardInterrupt:
        for process in processes:
            if process.poll() is None:
                process.terminate()
        raise
    return_codes = [process.wait() for process in processes]
    if return_codes != [0, 0]:
        raise RuntimeError(f'Dual-GPU workers returned {return_codes}')

    subprocess.check_call([
        sys.executable, str(PIPELINE), 'finalize-shards',
        '--input-root', str(VALIDATION_DIR), '--output-dir', str(FULL_OUTPUT),
        '--num-shards', '2', '--modalities', 'T1W', 'T2W', 'T2FLAIR',
        '--report', str(REPORT_DIR / 'full_output_validation.json'),
    ])
else:
    print('Full dual-T4 sweep is OFF.')

## Build and validate `task3.zip` with the official builder

This is intentionally locked until you review the live rules. It refuses stale build trees or archives instead of silently overwriting them. The helper checks the canonical 180-case manifest, calls the official builder with `--tasks task3`, validates the exact 60-pair/180-file slab tree and every NIfTI, then checks the archive root and SHA-256.

In [ ]:
BUILD_PRETRAINED_TASK3_ZIP = False
if BUILD_PRETRAINED_TASK3_ZIP:
    if not LIVE_RULES_REVIEWED:
        raise RuntimeError('Review the live rules, then set LIVE_RULES_REVIEWED=True.')
    subprocess.check_call([
        sys.executable, str(SUBMISSION_HELPER),
        '--official-repo', str(OFFICIAL_REPO),
        '--predictions-dir', str(FULL_OUTPUT),
        '--validation-input-root', str(VALIDATION_DIR),
        '--output-dir', str(SUBMISSION_DIR), '--archive', str(ARCHIVE),
        '--report', str(REPORT_DIR / 'task3_submission_validation.json'),
        '--expected-official-commit', OFFICIAL_COMMIT,
        '--expected-checkpoint-sha256', EXPECTED_CHECKPOINT_SHA256,
    ])
    final_report = json.loads(
        (REPORT_DIR / 'task3_submission_validation.json').read_text(encoding='utf-8')
    )
    print(json.dumps(final_report['archive'], indent=2))
    print('Upload was not performed. Download the archive from Kaggle output files.')
else:
    print('Submission build is OFF.')

## Result

After the final cell passes, download `/kaggle/working/mrixfields-task3/task3.zip` and keep the JSON reports under `results/reports/` with it. Kaggle exposes `/kaggle/working` files when you save a notebook version. This notebook never submits or uploads the archive.